# A method to get genome coordinates from a list of genes

https://bioinformatics.stackexchange.com/questions/17896/how-to-find-list-of-genomic-coordinates-from-list-of-genes-using-ucsc-website


In [ ]:
conda install bioconda::bedops  # for convert2bed

srun --partition=aoraki --ntasks=1 --cpus-per-task=8 --mem=16G --time=4:00:00 --pty bash -i

wget -qO- ftp://ftp.ebi.ac.uk/pub/databases/gencode/Gencode_human/release_38/gencode.v38.annotation.gff3.gz \
    | gunzip --stdout - \
    | awk '$3 == "gene"' - \
    | convert2bed -i gff - \
    > all_genes_hg38.bed

awk -F '\t' 'BEGIN { print "chromosome\tstart\tend\tgene_id\tscore\tstrand\tsource\ttype\tphase\tattributes" } NR == FNR { requested[$1] = 1; next } { n = split($10, attributes, ";"); for (i = 1; i <= n; i++) if (attributes[i] ~ /^gene_name=/) { name = substr(attributes[i], 11); if (name in requested) print; break } }' \
    genes_of_interest.txt all_genes_hg38.bed > genes_of_interest_coords.tsv

Manually tidy the table to just "chromosome,start,end,strand,gene_type,gene_name"  
gene type = protein_coding, lncRNA, etc.